---
## ✅ Key Findings Summary

| # | Finding | Business Impact |
|---|---------|----------------|
| 1 | **26.5 % churn rate** — dataset is imbalanced | Must use F1/AUC, NOT accuracy as primary metric |
| 2 | **Month-to-month contracts** → ~42 % churn rate | Contract type will be the #1 model feature |
| 3 | **Higher monthly charges** correlate with churn | Pricing strategy review needed |
| 4 | **Short tenure (< 12 months)** = highest risk window | Early engagement campaigns critical |
| 5 | **Fiber optic** customers churn more than DSL | Service quality / promise vs reality gap |
| 6 | **Lack of add-on services** (security, support, backup) increases churn | Bundle offers could reduce churn |

**Next step:** Feature engineering → build 10 new features that capture these business insights → `02_feature_engineering.ipynb`

In [ ]:
# ── Train/Test split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = split_data(df)

# Save processed train/test (before feature engineering) for reference
train_df = X_train.copy(); train_df["Churn"] = y_train.values
test_df  = X_test.copy();  test_df["Churn"]  = y_test.values

import os
os.makedirs("../data/processed", exist_ok=True)
train_df.to_csv("../data/processed/train_raw.csv", index=False)
test_df.to_csv("../data/processed/test_raw.csv",  index=False)
print("✅ Raw train/test splits saved to data/processed/")

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle("EDA Summary Dashboard — Telco Churn", fontsize=16, fontweight="bold", y=1.01)

# 1. Class distribution
ax1 = fig.add_subplot(2, 3, 1)
churn_counts.plot.bar(ax=ax1, color=sns.color_palette("Set2")[:2], edgecolor="white", rot=0)
ax1.set_title("1. Class Distribution", fontweight="bold")
ax1.set_xlabel(""); ax1.set_ylabel("Count")
for bar, pct in zip(ax1.patches, churn_pct.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f"{pct:.1f}%", ha="center", fontweight="bold")

# 2. Churn rate by Contract
ax2 = fig.add_subplot(2, 3, 2)
plot_churn_rate_by_cat(df, "Contract", ax2)

# 3. Churn rate by InternetService
ax3 = fig.add_subplot(2, 3, 3)
plot_churn_rate_by_cat(df, "InternetService", ax3)

# 4. Tenure distribution
ax4 = fig.add_subplot(2, 3, 4)
for churn_val, color in palette.items():
    df[df["Churn"] == churn_val]["tenure"].plot.hist(
        bins=25, alpha=0.6, color=color, ax=ax4, label=labels[churn_val])
ax4.set_title("4. Tenure Distribution by Churn", fontweight="bold")
ax4.set_xlabel("Tenure (months)"); ax4.set_ylabel("Count"); ax4.legend()

# 5. Churn rate by TechSupport
ax5 = fig.add_subplot(2, 3, 5)
plot_churn_rate_by_cat(df, "TechSupport", ax5)

# 6. Monthly charges by churn
ax6 = fig.add_subplot(2, 3, 6)
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", palette="Set2",
            hue="Churn", legend=False, ax=ax6)
ax6.set_title("6. Monthly Charges vs Churn", fontweight="bold")
ax6.set_xlabel("Churn (0=No, 1=Yes)")

plt.tight_layout()
plt.savefig("../reports/fig_07_summary_dashboard.png", bbox_inches="tight", dpi=150)
plt.show()

---
## Section 6 — Summary Dashboard & Key Findings

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── 1. Churn rate: Contract × InternetService ─────────────────────────────────
pivot1 = df.groupby(["Contract", "InternetService"])["Churn"].mean().unstack() * 100
pivot1.plot(kind="bar", ax=axes[0], colormap="Set2", edgecolor="white", rot=15)
axes[0].set_title("Churn Rate:\nContract × Internet Service", fontweight="bold")
axes[0].set_xlabel("Contract Type")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].legend(title="Internet Service", fontsize=8)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# ── 2. Churn rate by tenure bucket ────────────────────────────────────────────
df["tenure_bucket"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72],
                              labels=["0–12 mo", "13–24 mo", "25–48 mo", "49–72 mo"],
                              include_lowest=True)
tenure_churn = df.groupby("tenure_bucket", observed=True)["Churn"].mean() * 100
bars = axes[1].bar(tenure_churn.index, tenure_churn.values,
                   color=sns.color_palette("Set2", 4), edgecolor="white")
for bar, val in zip(bars, tenure_churn.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.5, f"{val:.1f}%",
                 ha="center", va="bottom", fontweight="bold")
axes[1].set_title("Churn Rate by Tenure Bucket", fontweight="bold")
axes[1].set_xlabel("Tenure Group")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

# ── 3. Monthly charges KDE: churners vs non-churners ─────────────────────────
for churn_val, color in palette.items():
    subset = df[df["Churn"] == churn_val]["MonthlyCharges"]
    subset.plot.kde(ax=axes[2], color=color, label=labels[churn_val], linewidth=2)
axes[2].set_xlabel("Monthly Charges ($)")
axes[2].set_ylabel("Density")
axes[2].set_title("Monthly Charges Distribution\n(Churners vs Non-Churners)", fontweight="bold")
axes[2].legend()
axes[2].fill_between(axes[2].lines[0].get_xdata(),
                     axes[2].lines[0].get_ydata(), alpha=0.1,
                     color=list(palette.values())[0])
axes[2].fill_between(axes[2].lines[1].get_xdata(),
                     axes[2].lines[1].get_ydata(), alpha=0.1,
                     color=list(palette.values())[1])

plt.tight_layout()
plt.savefig("../reports/fig_06_feature_interactions.png", bbox_inches="tight")
plt.show()

# Clean up temp column
df.drop("tenure_bucket", axis=1, inplace=True)

---
## Section 5 — Feature Interactions

**💡 Key Insights from Categorical Features:**
- **Contract type** is likely the strongest predictor — month-to-month customers churn at ~42 %, vs ~3 % for two-year contracts.
- **Fiber optic** internet service has ~42 % churn vs ~19 % for DSL — suggests service quality or pricing issues.
- **Electronic check** payment method correlates with ~45 % churn vs 15–18 % for other methods.
- Customers **without** OnlineSecurity, TechSupport, or OnlineBackup churn significantly more.

In [ ]:
def plot_churn_rate_by_cat(df, col, ax, palette="Set2"):
    """Horizontal bar chart of churn rate (%) per category."""
    rates = (df.groupby(col)["Churn"].mean() * 100).sort_values(ascending=True)
    colors = sns.color_palette(palette, len(rates))
    bars = ax.barh(rates.index, rates.values, color=colors, edgecolor="white")
    for bar, val in zip(bars, rates.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=9)
    ax.set_xlabel("Churn Rate (%)")
    ax.set_title(f"Churn Rate by {col}", fontweight="bold")
    ax.set_xlim(0, rates.max() * 1.2)
    return rates

# ── High-impact categorical features ─────────────────────────────────────────
highlight_cols = ["Contract", "InternetService", "PaymentMethod",
                  "OnlineSecurity", "TechSupport", "OnlineBackup"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, highlight_cols):
    plot_churn_rate_by_cat(df, col, ax)

plt.suptitle("Churn Rate by Categorical Features", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../reports/fig_05_categorical_churn_rates.png", bbox_inches="tight")
plt.show()

---
## Section 4 — Categorical Feature Analysis

For each categorical feature we compute the **churn rate per category** — this shows which categories are highest risk.

In [ ]:
# ── Correlation matrix (numerical + target) ───────────────────────────────────
corr_cols = num_cols + ["Churn"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, ax=axes[0],
            linewidths=0.5, square=True)
axes[0].set_title("Correlation Matrix\n(Numerical Features + Target)", fontweight="bold")

# ── Scatter: tenure vs MonthlyCharges coloured by Churn ───────────────────────
scatter_data = df.sample(min(2000, len(df)), random_state=42)   # sample for speed
for churn_val, color in palette.items():
    mask = scatter_data["Churn"] == churn_val
    axes[1].scatter(scatter_data.loc[mask, "tenure"],
                    scatter_data.loc[mask, "MonthlyCharges"],
                    alpha=0.4, s=15, color=color, label=labels[churn_val])
axes[1].set_xlabel("Tenure (months)")
axes[1].set_ylabel("Monthly Charges ($)")
axes[1].set_title("Tenure vs Monthly Charges\n(coloured by Churn)", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/fig_04_correlation_scatter.png", bbox_inches="tight")
plt.show()

**💡 Key Insight:**  
Churners have **higher MonthlyCharges** but **lower TotalCharges** — meaning they pay more per month but haven't been customers long.  
This makes intuitive sense: expensive plans → dissatisfaction → churn early.

In [ ]:
# ── Box plots grouped by Churn ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x="Churn", y=col, ax=ax,
                palette="Set2", hue="Churn", legend=False)
    ax.set_title(f"{col} by Churn", fontweight="bold")
    ax.set_xlabel("Churn (0=No, 1=Yes)")
    ax.set_ylabel(col)
plt.suptitle("Numerical Features — Box Plots by Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_03_numerical_boxplots.png", bbox_inches="tight")
plt.show()

# ── Key insight ───────────────────────────────────────────────────────────────
churn_means = df.groupby("Churn")[num_cols].mean()
print("Mean values by churn label:")
display(churn_means.rename(index={0: "No Churn", 1: "Churn"}))

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
palette  = {0: sns.color_palette("Set2")[0], 1: sns.color_palette("Set2")[1]}
labels   = {0: "No Churn", 1: "Churn"}

# ── Histograms ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, num_cols):
    for churn_val, color in palette.items():
        subset = df[df["Churn"] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=labels[churn_val], edgecolor="white")
    ax.set_title(f"Distribution of {col}", fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend()
plt.suptitle("Numerical Feature Distributions by Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_02_numerical_histograms.png", bbox_inches="tight")
plt.show()

---
## Section 3 — Numerical Feature Analysis

Three numerical features: `tenure`, `MonthlyCharges`, `TotalCharges`.

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_pct    = df["Churn"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Bar chart ─────────────────────────────────────────────────────────────────
bars = axes[0].bar(["No Churn (0)", "Churn (1)"], churn_counts.values,
                   color=sns.color_palette("Set2")[:2], edgecolor="white", width=0.5)
for bar, pct in zip(bars, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 50, f"{pct:.1f}%",
                 ha="center", va="bottom", fontweight="bold", fontsize=12)
axes[0].set_title("Customer Churn Distribution", fontweight="bold")
axes[0].set_ylabel("Number of Customers")
axes[0].set_ylim(0, churn_counts.max() * 1.15)

# ── Pie chart ─────────────────────────────────────────────────────────────────
axes[1].pie(churn_counts.values, labels=["No Churn", "Churn"],
            autopct="%1.1f%%", colors=sns.color_palette("Set2")[:2],
            startangle=90, explode=(0, 0.05),
            textprops={"fontsize": 12})
axes[1].set_title("Churn Proportion", fontweight="bold")

plt.suptitle("Target Variable — Class Imbalance Overview", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../reports/fig_01_target_distribution.png", bbox_inches="tight")
plt.show()

print(f"\nClass distribution:\n  No Churn: {churn_counts[0]:,} ({churn_pct[0]:.1f}%)")
print(f"  Churn:    {churn_counts[1]:,} ({churn_pct[1]:.1f}%)")
print(f"\n⚠️  Imbalance ratio: {churn_counts[0]/churn_counts[1]:.1f}:1")

---
## Section 2 — Target Variable Analysis (Most Important!)

> **The dataset is imbalanced — ~73 % No-Churn, ~27 % Churn.**  
> This means **accuracy alone is completely misleading**.  
> A model that always predicts "No Churn" would score **73 % accuracy** while catching **zero** actual churners.  
> We will use **F1-score, Precision, Recall, and ROC-AUC** as our primary metrics.

In [ ]:
# ── Clean data ────────────────────────────────────────────────────────────────
df = clean_data(df_raw)
col_types = identify_column_types(df)

print(f"\nColumn categories:")
for k, v in col_types.items():
    print(f"  {k:12s}: {v}")

In [ ]:
print("── Numerical summary ──")
display(df_raw.describe())
print("\n── Categorical / object summary ──")
display(df_raw.describe(include="object"))

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df_raw = load_data(RAW_PATH)

print(f"Shape: {df_raw.shape}  →  {df_raw.shape[0]:,} customers × {df_raw.shape[1]} columns")
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")
print(f"\nNull values per column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
df_raw.head()

---
## Section 1 — Load & Initial Inspection

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so we can import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── consistent style ──────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13,
                     "axes.labelsize": 11, "xtick.labelsize": 10,
                     "ytick.labelsize": 10})

from src.preprocessing import load_data, clean_data, identify_column_types, split_data

print("✅ Imports OK")

# 01 — EDA & Preprocessing
## ML Classification & Prediction System | Telco Customer Churn

**Goal:** Deeply understand the data before modelling.  
Good EDA is what separates data scientists from people who just run `model.fit()`.

---
**What we cover in this notebook:**
1. Load & initial inspection
2. Target variable analysis (class imbalance)
3. Numerical feature analysis
4. Categorical feature analysis
5. Feature interactions
6. Summary dashboard + key findings